# Lab 1: Two files, one truth
*Python for Healthcare · Stanford Technology Training (ITS-2519) · Dr. Lionel Levine*
**Session 1 · 35 minutes: about 27 minutes of work, then we debrief together**

**The study (fictional).** The Home BP Coaching Trial at Kestrel Valley Health randomized 240 adults with
uncontrolled hypertension (systolic BP of 140 or higher) to 12 weeks of nurse-led home BP coaching or to usual care,
at two clinics, North and South. All data are synthetic.

**The files**
- `enrollment.csv`: one row per randomized participant
- `visits.csv`: one row per clinic visit, at weeks 0, 4, 8, and 12 (a missed visit has no row)

**Your job.** Build one clean row per participant, ready for analysis, and prove it's right. You'll start by running
*stop rules*: checks that make the notebook refuse to go on until the data are right. Then you'll find and fix the
problems until every rule passes.

**Using Gemini.** Each fix comes with a suggested prompt. Gemini writes the code; you decide whether it's right.

### Data dictionary

**enrollment.csv**

| Column | Meaning |
|---|---|
| `participant_id` | Study ID, KV1001 to KV1240 |
| `site` | Clinic: North or South |
| `arm` | Randomized group: coaching or usual care |
| `randomization_date` | Date randomized (YYYY-MM-DD) |
| `age` | Age in years |
| `sex` | F or M |
| `bmi` | Body mass index (kg/m²); blank if not recorded |
| `diabetes` | yes or no |
| `smoker` | Current smoker, yes or no; blank if not asked |
| `n_bp_meds` | Number of BP medications at baseline |

**visits.csv**

| Column | Meaning |
|---|---|
| `participant_id` | Study ID |
| `visit_week` | 0 (baseline), 4, 8, or 12 |
| `visit_date` | Date of the visit |
| `sbp`, `dbp` | Systolic and diastolic office BP (mmHg) |
| `weight` | Weight (kg) |

## Step 0. Load the files (1 minute)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_URL = "https://raw.githubusercontent.com/lionellevine/python-for-healthcare/main/data/session1/"

def load(name):
    """Read a course file: your uploaded copy if there is one, otherwise the course repo."""
    return pd.read_csv(name) if Path(name).exists() else pd.read_csv(DATA_URL + name)

enrollment = load("enrollment.csv")
visits = load("visits.csv")
print("enrollment:", enrollment.shape)
print("visits:    ", visits.shape)

## Step 1. Run the stop rules (2 minutes)
These are the rules the data must pass before anyone analyzes them. Read each rule, then run the cell.
Don't expect it to pass yet.

In [ ]:
def check(rule, passed):
    print(("PASS  " if passed else "FAIL  ") + rule)
    return bool(passed)

def stop_rules(enrollment, visits):
    """Print every rule, and return True only if all of them pass."""
    readings = visits.dropna(subset=["sbp", "dbp"])
    results = [
        check("Exactly 240 randomized participants", len(enrollment) == 240),
        check("Every participant_id appears once", enrollment["participant_id"].is_unique),
        check("Arm is exactly 'coaching' or 'usual care'",
              set(enrollment["arm"]) == {"coaching", "usual care"}),
        check("No 999 codes remain in SBP or DBP", not (visits[["sbp", "dbp"]] == 999).any().any()),
        check("Every SBP is plausible (70 to 260)", readings["sbp"].between(70, 260).all()),
        check("SBP is higher than DBP in every reading", (readings["sbp"] > readings["dbp"]).all()),
    ]
    return all(results)

stop_rules(enrollment, visits)

**Expected: every line says FAIL.** The notebook isn't broken; the data are. Your job is to turn every line into
PASS without changing anything that's true.

## Step 2. Look at enrollment (4 minutes)
Run the three cells, then answer the questions below them.

In [ ]:
print(len(enrollment), "rows,", enrollment["participant_id"].nunique(), "unique IDs")
enrollment[enrollment["participant_id"].duplicated(keep=False)]

In [ ]:
enrollment["arm"].value_counts()

In [ ]:
enrollment.isna().sum()

1. Which participant appears twice? Are the two rows identical?
2. How many spellings of `arm` are there? How many should there be?
3. Two columns have blanks. Check the data dictionary: are those blanks errors, or real missing information?
   Should you "fix" them?

## Step 3. Fix enrollment (5 minutes)
**3a. Remove the exact duplicate row.**
Suggested prompt: *Remove exact duplicate rows from `enrollment`, save the result back to `enrollment`, and print
the row count before and after.* (Or write it yourself: `enrollment = enrollment.drop_duplicates()`)

In [ ]:
# 3a. Your code (or Gemini's) goes here

**3b. Standardize the arm labels** so there are exactly two values.
Suggested prompt: *Standardize the `arm` column in `enrollment` so the only values are "coaching" and
"usual care", save it back to `enrollment`, and show `value_counts()` before and after.*
(The pattern: `.str.strip().str.lower()`)

In [ ]:
# 3b. Your code (or Gemini's) goes here

**3c. Leave the blanks in `bmi` and `smoker` alone.** They're real missing information, not errors.
If an assistant offers to drop those rows, say no: that would remove randomized participants from the trial.

Now rerun the stop rules. The three enrollment rules should pass.

In [ ]:
stop_rules(enrollment, visits)

## Step 4. Look at the visits (4 minutes)
**Why are there fewer visits at weeks 8 and 12? Is that an error?**

In [ ]:
visits["visit_week"].value_counts().sort_index()

Look at the maximum values. What could an SBP of 999 mean?

In [ ]:
visits[["sbp", "dbp"]].describe().round(1)

In [ ]:
visits[(visits["sbp"] >= 900) | (visits["dbp"] >= 900)]

These rows have a diastolic reading higher than the systolic one. Look at the numbers: what probably happened?

In [ ]:
visits[visits["dbp"] > visits["sbp"]]

## Step 5. Fix the visits, and note each decision (5 minutes)
Every fix is a decision. In a real trial, the study team decides, writes the decision down, and sends a data query
to the site. For today, make these two decisions.

**5a. 999 is a missing-value code, not a reading.** Replace it with a true blank (`np.nan`) in both `sbp` and `dbp`.
Suggested prompt: *In `visits`, replace 999 in both `sbp` and `dbp` with NaN, save the result back to `visits`,
and print how many values changed.*

In [ ]:
# 5a. Your code (or Gemini's) goes here

**5b. Swap SBP and DBP back** where the diastolic value is higher than the systolic value. This cell is written
for you: read it line by line, then run it.

Why `.to_numpy()`? Without it, pandas lines the values up by column name and quietly puts each value back where
it came from, so nothing swaps. It's a classic bug in AI-written code, and the last stop rule would catch it.

In [ ]:
swapped = visits["dbp"] > visits["sbp"]        # True for each reading where DBP is higher
print("swapped readings:", swapped.sum())
visits.loc[swapped, ["sbp", "dbp"]] = visits.loc[swapped, ["dbp", "sbp"]].to_numpy()
visits[swapped]

**Note your decisions.** We'll fill this in together in the debrief. Double-click to edit.

| Problem | How many | Decision | Why |
|---|---|---|---|
| Duplicate row |  |  |  |
| Arm spellings |  |  |  |
| 999 codes |  |  |  |
| Swapped SBP and DBP |  |  |  |

Rerun the stop rules. All six should pass.

In [ ]:
stop_rules(enrollment, visits)

## Step 6. One row per participant (5 minutes)
You need a table with one row per participant: the enrollment information plus baseline (week 0) and week-12 BP.

**Predict first.** `enrollment` has 240 rows and `visits` has a bit over 900. If you merge them on `participant_id`,
how many rows do you get?

In [ ]:
naive = enrollment.merge(visits, on="participant_id")
print(len(naive), "rows")

Each participant now appears once per visit. That's right for some analyses, but not for a table that should have
one row per person. The fix is to reshape `visits` so each participant has one row with a column per visit,
which is called a *pivot*. Read this cell line by line before you run it.

In [ ]:
# One row per participant, with a column for each visit's SBP and DBP
wide = visits.pivot(index="participant_id", columns="visit_week", values=["sbp", "dbp"])
# The pivot names its columns like ("sbp", 0). The next line renames them to sbp_w0, sbp_w12, and so on.
wide.columns = [f"{measure}_w{week}" for measure, week in wide.columns]
wide = wide[["sbp_w0", "dbp_w0", "sbp_w12", "dbp_w12"]].reset_index()
wide.head()

In [ ]:
analysis = enrollment.merge(wide, on="participant_id", how="left", validate="one_to_one")
print(analysis.shape)

**The protected stop rule.** This one must pass before anyone analyzes the table.

In [ ]:
assert len(analysis) == 240, f"Expected 240 rows (one per participant), found {len(analysis)}"
assert analysis["participant_id"].is_unique, "A participant appears more than once"
print("✅ One row per randomized participant:", len(analysis))

## Step 7. The final gate (2 minutes)
The gate refuses to pass until every stop rule does. Then compare your table with the course checkpoint
(`analysis_s1.csv`), which Lab 2 uses.

In [ ]:
assert stop_rules(enrollment, visits), "Fix the FAIL lines above before you go on"
print("✅ All stop rules pass")

In [ ]:
checkpoint = load("analysis_s1.csv")
print("Same number of participants:     ", len(analysis) == len(checkpoint))
print("Same number with a week-12 SBP:  ",
      analysis["sbp_w12"].notna().sum() == checkpoint["sbp_w12"].notna().sum())
mine = analysis.groupby("arm")["sbp_w12"].mean().round(1)
ref = checkpoint.groupby("arm")["sbp_w12"].mean().round(1)
print("Same week-12 mean SBP by arm:    ", mine.equals(ref))

**Why it matters.** We'll run this together in the debrief: it compares the week-12 averages in the raw file with
your cleaned table.

In [ ]:
raw_w12 = (load("visits.csv").query("visit_week == 12")
           .merge(enrollment[["participant_id", "arm"]], on="participant_id"))
print("Week-12 mean SBP by arm, raw file:")
print(raw_w12.groupby("arm")["sbp"].mean().round(1), "\n")
print("Week-12 mean SBP by arm, cleaned:")
print(analysis.groupby("arm")["sbp_w12"].mean().round(1))

## If you finish early: two bonus checks
1. Compare the median `weight` at each clinic. Does one clinic look like it recorded pounds instead of kilograms?
2. Is any visit dated before that participant's randomization date?

Don't change the data. For each one, write the data query you'd send to the site.

In [ ]:
# Bonus: your code (or Gemini's) goes here

## Before the debrief, jot down
1. Which problem would have changed the trial's result the most if you'd missed it?
2. Which of your fixes were decisions rather than facts?
3. Write one more stop rule this notebook should enforce.